In [ ]:
import requests as req
import pandas as pd
import os, json
import pooch
from bs4 import BeautifulSoup as bs

In [ ]:
# 建立儲存圖片的資料夾，不存在就新增
folder_path = 'line_stickers'
os.makedirs(folder_path, exist_ok=True)

In [ ]:
 #放貼圖資訊用
li_stickers = []

# LINE 貼圖的網址
url = 'https://store.line.me/stickershop/product/38043/zh-Hant'

# 自訂標頭
my_headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36'
}

# 發出 GET 請求
res = req.get(url, headers=my_headers)

# 建立 soup 物件
soup = bs(res.text, 'lxml')

'''
備註:
1. soup.select()：回傳的結果是元素集合（list 型態，BeautifulSoup ResultSet）
2. soup.select_one()：回傳的結果是單一元素（BeautifulSoup Result）
'''
# 取得放置貼圖的 li 元素 (list 型態)
li_elms = soup.select(
    "ul.mdCMN09Ul.FnStickerList > li.mdCMN09Li.FnStickerPreviewItem"
)


# 逐一取得 li 元素中的 data-preview 資訊
for li in li_elms:    
    # 取得 data-preview 屬性的值（字串），再轉成物件 
    obj = json.loads( li['data-preview'] ) # 另一種寫法：li.get("data-preview")
    
    # 將重要資訊放置在 list 當中，幫助我們稍候進行資料下載與儲存
    li_stickers.append(obj)

# 將圖片資訊轉成 DataFrame，然後存成 Excel 檔案
df_stickers = pd.DataFrame(li_stickers)
df_stickers.to_excel("line_stickers.xlsx", index=False)

In [ ]:
# 下載圖片
for obj in li_stickers: 
    saved_path = pooch.retrieve(
        url=obj['staticUrl'],
        path=folder_path,
        progressbar=True,
        # fname=obj['id'] + '.png',
    )
    print(f"Save to {saved_path}")